Resources
- https://medium.com/@ashishgupta_65016/constructing-visual-hull-for-3d-object-reconstruction-1552aded5b8


In [41]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from skimage import measure

In [ ]:
# Parameters
VOXEL_RES = 64
IMAGE_FOLDER = "D:/Onedrive/experiments/experiments/3d_reconstruction/shape_from_silhouette/data/KKeishiro Shape-from-Silhouettes master data"
IMAGE_COUNT = 17
IMAGE_PREFIX = "david_"
IMAGE_EXT = ".jpg"
THRESH = 200

Helper Functions

In [45]:
def load_projection_matrix(pa_file):
    """Load projection matrix from .pa file with robust parsing"""
    with open(pa_file, 'r') as f:
        lines = f.readlines()
    
    # Extract all numerical values
    values = []
    for line in lines[1:]:  # Skip header line
        values.extend(map(float, line.split()))
    
    # Reshape into 3x4 matrix
    matrix = np.array(values).reshape(3, 4)
    return matrix

def binarize_image(image_path, threshold=128):
    """Convert image to binary silhouette"""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, binary = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)
    return binary

def create_voxel_grid(bounds, resolution=50):
    """Create a voxel grid within given bounds"""
    x = np.linspace(bounds[0], bounds[1], resolution)
    y = np.linspace(bounds[2], bounds[3], resolution)
    z = np.linspace(bounds[4], bounds[5], resolution)
    xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')
    voxels = np.ones_like(xx, dtype=bool)
    return xx, yy, zz, voxels

def project_voxels(voxels, P, image_shape):
    """Project voxels onto image plane using projection matrix P"""
    # Convert voxels to homogeneous coordinates
    voxel_coords = np.column_stack([voxels[0].flatten(), 
                                   voxels[1].flatten(), 
                                   voxels[2].flatten(), 
                                   np.ones(len(voxels[0].flatten()))])
    
    # Project to 2D (P is 3x4)
    projected = P @ voxel_coords.T
    projected = projected / projected[2, :]  # normalize by z
    
    # Reshape and scale to image coordinates
    u = np.round(projected[0, :]).astype(int)
    v = np.round(projected[1, :]).astype(int)
    
    # Create mask of valid projections (within image bounds)
    valid = (u >= 0) & (u < image_shape[1]) & (v >= 0) & (v < image_shape[0])
    
    return u, v, valid

def carve_voxels(voxels, silhouette, P, image_shape):
    """Carve voxels that don't project into the silhouette"""
    u, v, valid = project_voxels(voxels, P, image_shape)
    
    # Initialize mask (True means keep the voxel)
    mask = np.ones_like(voxels[3], dtype=bool)
    
    # For each voxel, check if it projects inside the silhouette
    for idx in np.ndindex(voxels[3].shape):
        flat_idx = np.ravel_multi_index(idx, voxels[3].shape)
        if valid[flat_idx]:
            if silhouette[v[flat_idx], u[flat_idx]] == 0:
                mask[idx] = False
    
    # Apply the mask
    voxels[3][~mask] = False
    
    return voxels

def visualize_visual_hull(voxels, step=None):
    """Visualize the current state of the visual hull"""
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Extract surface voxels using marching cubes
    if np.any(voxels[3]):
        try:
            verts, faces, _, _ = measure.marching_cubes(voxels[3].astype(np.float32), 0)
            
            # Scale vertices back to original coordinates
            verts[:, 0] = voxels[0].min() + verts[:, 0] * (voxels[0].max() - voxels[0].min()) / (voxels[0].shape[0] - 1)
            verts[:, 1] = voxels[1].min() + verts[:, 1] * (voxels[1].max() - voxels[1].min()) / (voxels[1].shape[0] - 1)
            verts[:, 2] = voxels[2].min() + verts[:, 2] * (voxels[2].max() - voxels[2].min()) / (voxels[2].shape[0] - 1)
            
            ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2], 
                           cmap='Spectral', lw=1, alpha=0.8)
        except RuntimeError:
            print(f"No isosurface found at step {step}")
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'Visual Hull {f"after step {step}" if step is not None else ""}')
    plt.tight_layout()
    plt.show()

def visualize_silhouette_and_contour(silhouette, step):
    """Visualize the silhouette and its contour"""
    plt.figure(figsize=(8, 6))
    plt.imshow(silhouette, cmap='gray')
    
    # Find contours
    contours, _ = cv2.findContours(silhouette, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Draw contours
    for contour in contours:
        plt.plot(contour[:, 0, 0], contour[:, 0, 1], 'r-', linewidth=2)
    
    plt.title(f'Silhouette and Contour (Step {step})')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def main():
    # Parameters
    num_images = 18  # david_00 to david_17
    bounds = [-1, 1, -1, 1, -1, 1]  # Initial bounding volume (adjust as needed)
    resolution = 50  # Voxel grid resolution
    image_folder = "D:/Onedrive/experiments/experiments/3d_reconstruction/shape_from_silhouette/data/KKeishiro Shape-from-Silhouettes master data"
    
    # Create voxel grid
    xx, yy, zz, voxels = create_voxel_grid(bounds, resolution)
    voxel_grid = (xx, yy, zz, voxels)
    
    # Process each image
    for i in range(num_images):
        # Load image and projection matrix
        img_path = os.path.join(image_folder, f'david_{i:02d}.jpg')
        pa_path = os.path.join(image_folder, f'david_{i:02d}.pa')
        
        if not os.path.exists(img_path) or not os.path.exists(pa_path):
            print(f"Warning: Missing file for step {i}")
            continue
        
        # Binarize image
        silhouette = binarize_image(img_path)
        
        # Load projection matrix
        P = load_projection_matrix(pa_path)
        
        # Visualize silhouette and contour
        visualize_silhouette_and_contour(silhouette, i)
        
        # Carve voxels
        voxel_grid = carve_voxels(voxel_grid, silhouette, P, silhouette.shape)
        
        # Visualize intermediate result
        if i % 3 == 0 or i == num_images - 1:  # Show every 3 steps and final
            visualize_visual_hull(voxel_grid, i)
    
    # Final visualization
    visualize_visual_hull(voxel_grid, "final")

In [47]:
if __name__ == "__main__":
    main()

ValueError: cannot reshape array of size 8 into shape (3,4)